# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides an interactive guide for loading, exploring, and visualizing the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and contains multiple record sets and fields for analysis.


In [ ]:
# Ensure mlcroissant is installed (uncomment if running in a new environment)
!pip install mlcroissant

## 1. Data Loading
Let's load the dataset metadata and prepare for exploration using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset with mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access metadata (Note: use as attributes, not dict)
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")

## 2. Data Overview

Explore available record sets and their `@id` values, fields, and columns. All references are made using the Croissant `@id` to ensure precise and unambiguous access to data assets.

The FAIR^2 dataset contains one main record set holding the proteomics table. Let's look at what's available in the schema.

In [ ]:
# Print all record set @id and their details
print("Available Record Sets (by @id):")
for rs in meta.record_sets:
    print(f"  - @id: {rs['@id']}, name: {rs.get('name','<none>')}, description: {rs.get('description','<none>')}")

# Look at the fields for each RecordSet
for rs in meta.record_sets:
    print(f"\nFields in RecordSet '@id={rs['@id']}':")
    for field in rs['field']:
        print(f"  - @id: {field['@id']:50} name: {field.get('name','<none>'):30} dataType: {field.get('dataType')}")

Let's print a sample of records from the main record set.

In [ ]:
# Peek at a few records in the primary record set using its @id.
# For this dataset, the main record set has @id 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json#table1-records'
main_recordset_id = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json#table1-records'

for i, record in enumerate(dataset.records(record_set=main_recordset_id)):
    print(record)
    if i >= 2:
        break

## 3. Data Extraction

Now, let's load the main proteomics record set into a pandas DataFrame for further processing. We reference the record set using its `@id`. If the dataset included additional record sets, they could be added similarly by their `@id`.

In [ ]:
# List of record set @id's for extraction
record_sets = [main_recordset_id]
dataframes = {}

for rsid in record_sets:
    records = list(dataset.records(record_set=rsid))
    dataframes[rsid] = pd.DataFrame(records)

df = dataframes[main_recordset_id]
print(f"Columns in DataFrame (main record set):\n{df.columns.tolist()}")
df.head()

## 4. Exploratory Data Analysis (EDA)

Let's perform some filtering and transformations using the dataset's fields.

**Identify suitable fields:**
- For numeric analyses, let's use the field `cr:table1.coverage_percent` (protein sequence coverage percentage) and filter for values greater than 20.
- We'll also look at the field `cr:table1.abundance_norm` for normalization.

Field `@id`s used (from metadata):
- Coverage percent (as @id): `cr:table1.coverage_percent`
- Abundance norm (as @id): `cr:table1.abundance_norm`
- Grouping field: `cr:table1.sample_group` (if present)


In [ ]:
numeric_field = 'cr:table1.coverage_percent'   # field @id for coverage percent
abundance_field = 'cr:table1.abundance_norm'   # field @id for normalized abundance
group_field = 'cr:table1.sample_group'         # e.g., grouping by sample group if present

threshold = 20

# Filter records with coverage_percent > threshold
if numeric_field in df.columns:
    filtered_df = df[df[numeric_field].astype(float) > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}: {len(filtered_df)} records")

    # Normalize 'abundance_norm' if present
    if abundance_field in filtered_df.columns:
        filtered_df[f"{abundance_field}_zscore"] = (
            filtered_df[abundance_field].astype(float) - filtered_df[abundance_field].astype(float).mean()
        ) / filtered_df[abundance_field].astype(float).std()
        print(filtered_df[[numeric_field, abundance_field, f"{abundance_field}_zscore"]].head())
    else:
        print("abundance_norm field not found.")

    # Group by group_field if available
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"\nGrouped mean values by {group_field}:")
        print(grouped_df.head())
else:
    print(f"Field {numeric_field} not found in DataFrame.")

## 5. Visualization

We'll create visualizations of the coverage percent and normalized abundance distributions, using only records with coverage percent above our threshold.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field in df.columns:
    # Coverage percent histogram
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[numeric_field].astype(float), bins=30, color='skyblue', kde=True)
    plt.title(f"Distribution of {numeric_field} (> {threshold})")
    plt.xlabel("Coverage Percent")
    plt.ylabel("Count")
    plt.show()

    # Normalized abundance scatter
    if abundance_field in filtered_df.columns:
        plt.figure(figsize=(8,4))
        sns.scatterplot(x=filtered_df[numeric_field].astype(float), y=filtered_df[abundance_field].astype(float))
        plt.title(f"{abundance_field} vs {numeric_field}")
        plt.xlabel("Coverage Percent")
        plt.ylabel("Normalized Abundance")
        plt.show()


## 6. Conclusion

- We loaded and explored the FAIR^2 mass spectrometry dataset using the Croissant schema and `mlcroissant`.
- We identified fields by their Croissant `@id` and extracted the main record set as a DataFrame.
- Numeric filtering and normalization were performed on the coverage percentage and normalized abundance, and group-wise analysis was demonstrated (where available).
- Visualizations provide an overview of protein coverage and abundance distribution, useful for further biological insights.